# DSAI 490 — Assignment 1
## Representation Learning with AE & VAE on Medical MNIST

This notebook trains:
- **12 per-class models** (one AE + one VAE for each of 6 anatomical regions)
- **2 global models** (one AE + one VAE on all classes combined, used for cross-class latent-space visualization)

All code lives in the `src/` Python package committed to this repo; the notebook only orchestrates.

## 1. Environment setup

In [ ]:
# Colab: mount Drive and clone the repo. Skip these cells when running locally.
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print('Running locally, not in Colab.')

In [ ]:
REPO_URL = 'https://github.com/a7mdka7la/Assignment-1.git'
REPO_DIR = '/content/Assignment-1'

if IN_COLAB:
    import os, subprocess
    if not os.path.isdir(REPO_DIR):
        result = subprocess.run(
            ['git', 'clone', REPO_URL, REPO_DIR],
            capture_output=True, text=True,
        )
        print(result.stdout)
        print(result.stderr)
        if result.returncode != 0:
            raise RuntimeError(
                f'git clone failed (exit {result.returncode}). '
                'If the repo is private, generate a GitHub PAT and use '
                f"REPO_URL = 'https://<TOKEN>@github.com/a7mdka7la/Assignment-1.git'"
            )
    %cd $REPO_DIR

In [ ]:
%pip install -q -r requirements.txt

## 2. Dataset (Kaggle → Drive → extract)

In [ ]:
from src import data

if IN_COLAB:
    data.install_kaggle_credentials()
    data.download_dataset()

DATASET_ROOT = data.ensure_dataset()
print('Dataset root:', DATASET_ROOT)

In [ ]:
from src.config import CLASS_NAMES
from src.data import build_dataset, list_class_files
from src.viz import plot_class_samples

sample_datasets = {
    c: build_dataset(list_class_files(c, DATASET_ROOT)[:64], batch_size=16, shuffle=False)
    for c in CLASS_NAMES
}
plot_class_samples(sample_datasets, n_per_class=6)

## 3. Train all 14 models

In [ ]:
from src.utils import set_seed
from src.training import train_all
from src.config import TRAIN

set_seed()
runs, labeled_datasets = train_all(DATASET_ROOT, epochs=TRAIN.epochs, batch_size=TRAIN.batch_size)
print(f'Trained {len(runs)} models: {list(runs.keys())}')

## 4. Loss curves

In [ ]:
from src.viz import plot_loss_curves

histories = {tag: run.history for tag, run in runs.items()}
plot_loss_curves(histories)

## 5. Reconstructions (per class)

In [ ]:
from src.data import per_class_splits, build_dataset
from src.viz import plot_reconstructions_grid

splits = per_class_splits(DATASET_ROOT)
test_datasets = {
    c: build_dataset(splits[c][2], batch_size=32, shuffle=False)
    for c in CLASS_NAMES
}
ae_models = {c: runs[f'ae_{c}'].model for c in CLASS_NAMES}
vae_models = {c: runs[f'vae_{c}'].model for c in CLASS_NAMES}

plot_reconstructions_grid(ae_models, test_datasets,
                          filename='reconstructions_ae.png',
                          title='AE reconstructions per class')
plot_reconstructions_grid(vae_models, test_datasets,
                          filename='reconstructions_vae.png',
                          title='VAE reconstructions per class')

## 6. Latent space: cross-class view from the global models

In [ ]:
from src.viz import plot_latent_2d

ae_global = runs['ae_global'].model
vae_global = runs['vae_global'].model
test_labeled = labeled_datasets['test']

plot_latent_2d(ae_global, test_labeled, method='pca',
               filename='latent_ae_global_pca.png',
               title='Global AE latent space (PCA of 16-D codes)')
plot_latent_2d(vae_global, test_labeled, method='pca',
               filename='latent_vae_global_pca.png',
               title='Global VAE latent space (PCA of posterior means)')
plot_latent_2d(ae_global, test_labeled, method='tsne',
               filename='latent_ae_global_tsne.png',
               title='Global AE latent space (t-SNE of 16-D codes)')
plot_latent_2d(vae_global, test_labeled, method='tsne',
               filename='latent_vae_global_tsne.png',
               title='Global VAE latent space (t-SNE of posterior means)')

## 7. VAE sample generation and latent traversals

In [ ]:
from src.viz import plot_vae_samples, plot_latent_traversal

# Per-class VAE sampling grids.
for c in CLASS_NAMES:
    plot_vae_samples(vae_models[c], n=16,
                     filename=f'vae_samples_{c}.png',
                     title=f'{c} — samples from N(0, I)')

# Global VAE sampling grid (mix of all regions).
plot_vae_samples(vae_global, n=16,
                 filename='vae_samples_global.png',
                 title='Global VAE — samples from N(0, I)')

# Latent traversals for two dimensions of the global VAE.
plot_latent_traversal(vae_global, dim=0)
plot_latent_traversal(vae_global, dim=3)

## 8. Denoising comparison (AE vs. VAE)

In [ ]:
from src.viz import plot_denoising

for sigma in (0.1, 0.2, 0.3):
    plot_denoising(ae_global, test_datasets[CLASS_NAMES[0]], sigma=sigma,
                   filename=f'denoising_ae_sigma{sigma}.png',
                   title=f'AE denoising on {CLASS_NAMES[0]} at σ={sigma}')
    plot_denoising(vae_global, test_datasets[CLASS_NAMES[0]], sigma=sigma,
                   filename=f'denoising_vae_sigma{sigma}.png',
                   title=f'VAE denoising on {CLASS_NAMES[0]} at σ={sigma}')

## 9. Metrics summary (AE vs. VAE, per class)

In [ ]:
import pandas as pd
from src.utils import evaluate_reconstruction

rows = []
for c in CLASS_NAMES:
    mse_ae, ssim_ae = evaluate_reconstruction(ae_models[c], test_datasets[c])
    mse_vae, ssim_vae = evaluate_reconstruction(vae_models[c], test_datasets[c])
    rows.append({'class': c,
                 'AE MSE': mse_ae, 'AE SSIM': ssim_ae,
                 'VAE MSE': mse_vae, 'VAE SSIM': ssim_vae})
metrics_df = pd.DataFrame(rows).set_index('class')
metrics_df.to_csv('figures/metrics_per_class.csv')
metrics_df

## 10. Persist checkpoints to Drive (optional)

In [ ]:
import shutil, os
DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/dsai490_assignment1_checkpoints'
if IN_COLAB:
    os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
    for f in os.listdir('checkpoints'):
        if f.endswith('.weights.h5'):
            shutil.copy2(os.path.join('checkpoints', f), DRIVE_CHECKPOINT_DIR)
    print('Checkpoints copied to', DRIVE_CHECKPOINT_DIR)